In [1]:
!pip install -q -U transformers datasets accelerate evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.4 MB/s eta 0:00:00


In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    LEDForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

GPU available: True
Tesla T4


In [3]:
from google.colab import files
uploaded = files.upload()  # select archive__10_.zip from your computer

Saving archive (10).zip to archive (10).zip


In [4]:
import zipfile

zip_filename = list(uploaded.keys())[0]  # gets whatever name Colab actually saved it as
print("Using file:", zip_filename)

with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("bbc_data")

!find bbc_data -maxdepth 3 -type d

Using file: archive (10).zip
bbc_data
bbc_data/BBC News Summary
bbc_data/BBC News Summary/Summaries
bbc_data/BBC News Summary/Summaries/tech
bbc_data/BBC News Summary/Summaries/entertainment
bbc_data/BBC News Summary/Summaries/politics
bbc_data/BBC News Summary/Summaries/business
bbc_data/BBC News Summary/Summaries/sport
bbc_data/BBC News Summary/News Articles
bbc_data/BBC News Summary/News Articles/tech
bbc_data/BBC News Summary/News Articles/entertainment
bbc_data/BBC News Summary/News Articles/politics
bbc_data/BBC News Summary/News Articles/business
bbc_data/BBC News Summary/News Articles/sport
bbc_data/bbc news summary
bbc_data/bbc news summary/BBC News Summary
bbc_data/bbc news summary/BBC News Summary/Summaries
bbc_data/bbc news summary/BBC News Summary/News Articles


In [5]:
BASE_DIR = None
for root, dirs, _ in os.walk("bbc_data"):
    if "News Articles" in dirs and "Summaries" in dirs:
        BASE_DIR = root
        break
print("Found dataset at:", BASE_DIR)

articles_dir = os.path.join(BASE_DIR, "News Articles")
summaries_dir = os.path.join(BASE_DIR, "Summaries")

records = []
for category in os.listdir(articles_dir):
    art_cat_dir = os.path.join(articles_dir, category)
    sum_cat_dir = os.path.join(summaries_dir, category)
    if not os.path.isdir(art_cat_dir):
        continue
    for fname in os.listdir(art_cat_dir):
        art_path = os.path.join(art_cat_dir, fname)
        sum_path = os.path.join(sum_cat_dir, fname)
        if not os.path.exists(sum_path):
            continue
        with open(art_path, encoding="utf-8", errors="ignore") as f:
            article_text = f.read().strip()
        with open(sum_path, encoding="utf-8", errors="ignore") as f:
            summary_text = f.read().strip()
        if article_text and summary_text:
            records.append({"text": article_text, "summary": summary_text, "category": category})

df = pd.DataFrame(records)
print(df.shape)
print(df["category"].value_counts())
df.head()

Found dataset at: bbc_data/BBC News Summary
(2225, 3)
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,text,summary,category
0,Progress on new internet domains\n\nBy early 2...,By early 2005 the net could have two new domai...,tech
1,Blind student 'hears in colour'\n\nA blind stu...,Mr Wong has been blind from the age of seven a...,tech
2,"Millions to miss out on the net\n\nBy 2025, 40...",Although the percentage of Britons without hom...,tech
3,Tough rules for ringtone sellers\n\nFirms that...,"Christian Harris, partnership manager of mobil...",tech
4,Apple laptop is 'greatest gadget'\n\nThe Apple...,The Apple Powerbook 100 has been chosen as the...,tech


In [6]:
from sklearn.model_selection import train_test_split

SEED = 42
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["category"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["category"])

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 1780 | Val: 222 | Test: 223


In [7]:
MODEL_NAME = "allenai/led-base-16384"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = LEDForConditionalGeneration.from_pretrained(MODEL_NAME)
model.config.use_cache = False
model.gradient_checkpointing_enable()

print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/27.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/648M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/648M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Total parameters: 161,844,480


In [8]:
MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 256

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["text"], max_length=MAX_INPUT_LENGTH, truncation=True
    )
    labels = tokenizer(
        text_target=examples["summary"], max_length=MAX_TARGET_LENGTH, truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df[["text", "summary"]]).map(
    preprocess_function, batched=True, remove_columns=["text", "summary"]
)
val_ds = Dataset.from_pandas(val_df[["text", "summary"]]).map(
    preprocess_function, batched=True, remove_columns=["text", "summary"]
)
test_ds = Dataset.from_pandas(test_df[["text", "summary"]]).map(
    preprocess_function, batched=True, remove_columns=["text", "summary"]
)

print(train_ds)

Map:   0%|          | 0/1780 [00:00<?, ? examples/s]

Map:   0%|          | 0/222 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1780
})


In [9]:
base_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

class LEDDataCollator:
    def __init__(self, base_collator):
        self.base_collator = base_collator

    def __call__(self, features):
        batch = self.base_collator(features)
        global_attention_mask = torch.zeros_like(batch["attention_mask"])
        global_attention_mask[:, 0] = 1  # global attention on the first token
        batch["global_attention_mask"] = global_attention_mask
        return batch

data_collator = LEDDataCollator(base_collator)

In [10]:
import evaluate

rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 2) for k, v in result.items()}

    gen_lens = [np.count_nonzero(p != tokenizer.pad_token_id) for p in preds]
    result["gen_len"] = round(np.mean(gen_lens), 1)
    return result

In [16]:
training_args = Seq2SeqTrainingArguments(
    output_dir="results",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss


[transformers] Input ids are automatically padded from 219 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 966 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 548 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 377 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 191 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 512 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 416 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 178 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padde

KeyboardInterrupt: 

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_ds)
print("Test metrics (ROUGE-1 / ROUGE-2 / ROUGE-L / ROUGE-Lsum, gen_len):")
print(test_metrics)

[transformers] Input ids are automatically padded from 710 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 1016 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 746 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 932 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 656 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 576 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 291 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padded from 652 to 1024 to be a multiple of `config.attention_window`: 1024
[transformers] Input ids are automatically padd

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
0.986524,0.163127,3,67.070000,55.140000,48.000000,47.970000,178.100000


Test metrics (ROUGE-1 / ROUGE-2 / ROUGE-L / ROUGE-Lsum, gen_len):
{'eval_loss': 0.16312727332115173, 'eval_rouge1': 67.07, 'eval_rouge2': 55.14, 'eval_rougeL': 48.0, 'eval_rougeLsum': 47.97, 'eval_gen_len': 178.1}


In [ ]:
def summarize(text, max_input_length=MAX_INPUT_LENGTH, max_new_tokens=MAX_TARGET_LENGTH):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input_length).to(model.device)
    global_attention_mask = torch.zeros_like(inputs["input_ids"]).to(model.device)
    global_attention_mask[:, 0] = 1

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            global_attention_mask=global_attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=4,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

for i in range(3):
    row = test_df.iloc[i]
    print("ARTICLE:", row["text"][:300], "...")
    print("\nREFERENCE SUMMARY:", row["summary"])
    print("\nMODEL SUMMARY:", summarize(row["text"]))
    print("\n" + "=" * 80 + "\n")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ARTICLE: Kennedy criticises 'unfair' taxes

Gordon Brown has failed to tackle the "fundamental unfairness" in the tax system in his ninth Budget, Charles Kennedy has said.

How was it right that the poorest 20% of society were still paying more as a proportion of their income than the richest 20%, the Lib De ...

REFERENCE SUMMARY: Mr Kennedy added his party's priorities of free long-term care for the elderly, abolishing top-up fees and replacing the council tax would be funded by charging 50% income tax to those earning more than £100,000 per annum.The new £200 council tax rebate for pensioners did nothing to fix the "unfair tax", he added.The Lib Dem plan for a local income tax would benefit the typical household by more than £450 a year, with half of all pensioners paying no local tax and about three million being better off.On pensions, Mr Kennedy said it was a "scandal" that the system discriminated against women who had missed making National Insurance payments when they were hav

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL SUMMARY: The chancellor's announcement that he was to offer a £200 council tax rebate paid by pensioner households was merely a "sticking plaster" to a much bigger problem.The new £200 council tax rebate for pensioners did nothing to fix the "unfair tax", he added.The Lib Dem plan for a local income tax would benefit the typical household by more than £450 a year, with half of all pensioners paying no local tax and about three million being better off.On pensions, Mr Kennedy said it was a "scandal" that the system discriminated against women who had missed making National Insurance payments when they were having children.Gordon Brown has failed to tackle the "fundamental unfairness" in the tax system in his ninth Budget, Charles Kennedy has said.On pensions, Mr Kennedy said it was a "scandal" that the system discriminated against women who had missed making National Insurance payments when they were having children.


ARTICLE: Radcliffe yet to answer GB call

Paula Radcliffe has

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL SUMMARY: British team member Hayley Yelling said the team would understand if Radcliffe opted out of the event.Paula Radcliffe has been granted extra time to decide whether to compete in the World Cross-Country Championships.British team member Hayley Yelling said the team would understand if Radcliffe opted out of the event."It would be fantastic to have Paula in the team," said the European cross-country champion."But you have to remember that athletics is basically an individual sport and anything achieved for the team is a bonus.


ARTICLE: Attack prompts Bush site block

The official re-election site of President George W Bush is blocking visits from overseas users for "security reasons".

The blocking began early on Monday so those outside the US and trying to view the site got a message saying they are not authorised to view it. But ...

REFERENCE SUMMARY: There are now at least three working alternative domains for the Bush-Cheney campaign that let web users outside the 

In [ ]:
def summarize(text, max_input_length=MAX_INPUT_LENGTH, max_new_tokens=MAX_TARGET_LENGTH):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input_length).to(model.device)
    global_attention_mask = torch.zeros_like(inputs["input_ids"]).to(model.device)
    global_attention_mask[:, 0] = 1

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            global_attention_mask=global_attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=4,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

for i in range(3):
    row = test_df.iloc[i]
    print("ARTICLE:", row["text"][:300], "...")
    print("\nREFERENCE SUMMARY:", row["summary"])
    print("\nMODEL SUMMARY:", summarize(row["text"]))
    print("\n" + "=" * 80 + "\n")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ARTICLE: Kennedy criticises 'unfair' taxes

Gordon Brown has failed to tackle the "fundamental unfairness" in the tax system in his ninth Budget, Charles Kennedy has said.

How was it right that the poorest 20% of society were still paying more as a proportion of their income than the richest 20%, the Lib De ...

REFERENCE SUMMARY: Mr Kennedy added his party's priorities of free long-term care for the elderly, abolishing top-up fees and replacing the council tax would be funded by charging 50% income tax to those earning more than £100,000 per annum.The new £200 council tax rebate for pensioners did nothing to fix the "unfair tax", he added.The Lib Dem plan for a local income tax would benefit the typical household by more than £450 a year, with half of all pensioners paying no local tax and about three million being better off.On pensions, Mr Kennedy said it was a "scandal" that the system discriminated against women who had missed making National Insurance payments when they were hav

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL SUMMARY: The chancellor's announcement that he was to offer a £200 council tax rebate paid by pensioner households was merely a "sticking plaster" to a much bigger problem.The new £200 council tax rebate for pensioners did nothing to fix the "unfair tax", he added.The Lib Dem plan for a local income tax would benefit the typical household by more than £450 a year, with half of all pensioners paying no local tax and about three million being better off.On pensions, Mr Kennedy said it was a "scandal" that the system discriminated against women who had missed making National Insurance payments when they were having children.Gordon Brown has failed to tackle the "fundamental unfairness" in the tax system in his ninth Budget, Charles Kennedy has said.On pensions, Mr Kennedy said it was a "scandal" that the system discriminated against women who had missed making National Insurance payments when they were having children.


ARTICLE: Radcliffe yet to answer GB call

Paula Radcliffe has

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL SUMMARY: British team member Hayley Yelling said the team would understand if Radcliffe opted out of the event.Paula Radcliffe has been granted extra time to decide whether to compete in the World Cross-Country Championships.British team member Hayley Yelling said the team would understand if Radcliffe opted out of the event."It would be fantastic to have Paula in the team," said the European cross-country champion."But you have to remember that athletics is basically an individual sport and anything achieved for the team is a bonus.


ARTICLE: Attack prompts Bush site block

The official re-election site of President George W Bush is blocking visits from overseas users for "security reasons".

The blocking began early on Monday so those outside the US and trying to view the site got a message saying they are not authorised to view it. But ...

REFERENCE SUMMARY: There are now at least three working alternative domains for the Bush-Cheney campaign that let web users outside the 

In [ ]:
import pandas as pd

history = pd.DataFrame(trainer.state.log_history)
epoch_summary = history.groupby("epoch")[["loss", "eval_loss"]].mean(numeric_only=True)
print(epoch_summary)

In [ ]:
SAVE_DIR = "led_bbc_finetuned"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!zip -r led_bbc_finetuned.zip led_bbc_finetuned
from google.colab import files
files.download("led_bbc_finetuned.zip")